# polygon → MOC → shard → 3-D → numpy

The whole zagg read stack in five functions, one `%pip install`, and zero
credentials. A geojson polygon becomes a morton MOC; the MOC checks itself
against the store's own coverage; the covered shards open with timings; one
shard renders in 3-D (ATL03 + GEDI together); the current view exports to a
numpy tensor and saves to disk.

Everything below is reader-side and calls no zagg public API — `mortie` for
the geometry, `moczarr` for the store, plus the t-digest algebra that
`moczarr[zagg]` imports from zagg rather than vendoring (moczarr issue #19).
That algebra is the only zagg code on the path. It all runs anonymously
against public S3, binder-ready.

Its sibling is [`waveform_viewer.ipynb`](waveform_viewer.ipynb), which takes
the same polygon and stores down to the cell-level join: one GEDI o18
footprint against the 2×2 ATL03 o19 cells beneath it, both rebuilt from their
stored t-digests. The two are separate notebooks because this one needs
`%matplotlib widget` and that one `%matplotlib inline`; the backends collide
in a single kernel.

In [ ]:
%pip install -q mortie "moczarr[zagg]>=0.7" matplotlib ipympl ipywidgets
%matplotlib widget

import resource
import time
import zipfile
from io import BytesIO

import moczarr as mz
import numpy as np
from mortie import moc

# The drawing lives in viewers.py beside this notebook, so the cells below stay
# about the READ path. Both demo notebooks share it.
from viewers import BLOCK_ORDER, view3d

# One store per product, each appendable. Coverage answers which ground the
# store holds; the store name never does.
STORES = {
    "atl03": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/atl03_tdigest_o9.zarr",
        "19/h_tdigest_signal",
    ),
    "gedi": (
        "s3://us-west-2.opendata.source.coop/englacial/zagg/demo/gedi_flux_o9.zarr",
        "18/rx_flux",
    ),
}
S3 = {"region": "us-west-2", "anonymous": True}

rss = lambda: resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20  # noqa: E731

## One polygon in, covered shards out

Replace the polygon with any area within California or a NEON AOP site; the
cell tests whether each store actually covers it. An AOI in Alaska will pass
the ATL03 check and fail the GEDI one — GEDI flies on the ISS, so it sees no
higher than |lat| 51.6 and the Alaska NEON sites are outside its reach.

In [ ]:
aoi = {
    "features": [
        {
            "geometry": {
                "coordinates": [
                    [  # a ~4 km box on the SERC tract
                        [-76.56, 38.87],
                        [-76.50, 38.87],
                        [-76.50, 38.91],
                        [-76.56, 38.91],
                        [-76.56, 38.87],
                    ]
                ]
            }
        }
    ]
}

q = moc(aoi)
shards = None
for name, (root, _field) in STORES.items():
    assert mz.coverage_moc(root, **S3).contains(q), f"{name} does not contain the polygon"
    ids = set(mz.candidate_shards(root, aoi=q, **S3))
    shards = ids if shards is None else shards & ids
shards = sorted(shards)
shards

## Open one shard — every dataset, timed

In [ ]:
def open_shard(shard):
    """Return the open leaf handles for this shard, one per store.

    The cell exists primarily to show what reading a FULL shard costs -- the
    seconds, and the MB of resident memory. None of the data it reads is kept:
    the viewer below fetches per block on demand, which is what holds the
    notebook inside Binder's memory footprint when the sparse stored digests
    are cast to the dense tensors visualization needs.
    """
    handles = {}
    for name, (root, field) in STORES.items():
        t0, r0 = time.perf_counter(), rss()
        store = mz.open_leaf(root, shard, **S3)
        _, element = mz.open_ragged(store, field)
        n = sum(len(v) for _, v in mz.read_ragged(store, field))
        print(
            f"{name:6s} open+sweep {time.perf_counter() - t0:5.1f}s "
            f"(+{rss() - r0:4.0f} MB) — {n:,} centroids, element {element}"
        )
        handles[name] = (store, field)
    return handles


handles = open_shard(shards[0])

## The 3-D view — both sensors, exact centroids, time-aware

In [ ]:
view = view3d(handles, shards[0])

## Export what you see — a numpy tensor, shaped your way

In [ ]:
def export(view, sensor, n_bins=64, resolution=1.0, fit="degrade_resolution"):
    """The block on screen -> (rows, cols, n_bins) numpy tensor + z metadata.

    `n_bins` sets the vertical shape and `resolution` the z bin height you
    ASK FOR; the xy shape follows the sensor's cell order within the o12 block
    -- 2**(cell_order - 12) a side, so 128 for ATL03's o19 cells and 64 for
    GEDI's o18.

    `fit="degrade_resolution"` -- the default -- lets the store COARSEN z when
    a block's relief does not fit in `n_bins * resolution` metres, so the bin
    height that comes back is not always the one asked for. The print says
    which of the two happened instead of interpolating `dz` into a sentence
    that reads like the request was honoured.
    """
    store, field = handles[sensor]
    t, mask, (z0, dz), w = next(
        b
        for b in mz.read_tensors(
            store,
            field,
            n_bins=n_bins,
            resolution=resolution,
            block_order=BLOCK_ORDER,
            fit=fit,
            subtree=mz.morton_decimal(view.block),  # this block only, not the shard
        )
    )
    got = (
        "as asked"
        if abs(dz - resolution) < 1e-9
        else f"DEGRADED from the {resolution:g} m asked, to fit the relief in {n_bins} bins"
    )
    print(f"{sensor}: {t.shape} tensor, z = {z0:.1f} m + bin * {dz:g} m ({got})")
    return t, {"z0": z0, "dz": dz, "block": mz.morton_decimal(w)}


In [ ]:
gedi, meta = export(view, "gedi", n_bins=128, resolution=0.5)
np.save(f"gedi_{meta['block']}.npy", gedi)

atl03, meta = export(view, "atl03")  # asks for 64 bins at 1 m -- read what it says it got
np.save(f"atl03_{meta['block']}.npy", atl03)


In [ ]:
# ---- second export: every block of the shard as a 128 x 128 x 128 cube ----
# ATL03's cells are o19, so an o12 block is 2**(19-12) = 128 cells a side;
# ask for 128 z-bins at 1 m and the tensor is a cube. One `read_tensors`
# sweep yields one cube per populated block, so a shard exports a SET.
#
# `fit="degrade_resolution"` keeps a block whose trimmed relief exceeds 128 m
# rather than refusing it -- the z gain comes back per block, so the ones that
# did not fit at 1 m say so instead of silently pretending.
#
# Each cube is STREAMED into the archive as it arrives rather than collected
# and handed to `np.savez_compressed` in one call: 64 blocks x 8 MiB is ~512
# MiB resident at once, and mybinder.org caps the WHOLE container at 2 GB. An
# `.npz` is just a zip of `.npy` members, so `np.load` reads this back exactly
# the same way -- only the per-block metadata stays in memory.
path = f"atl03_cubes_{shards[0]}.npz"
cubes = {}
with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as zf:
    for tensor, mask, (z0, dz), w in mz.read_tensors(
        *handles["atl03"],
        n_bins=128,
        resolution=1.0,
        block_order=BLOCK_ORDER,
        fit="degrade_resolution",
    ):
        b = mz.morton_decimal(w)
        buf = BytesIO()
        np.save(buf, tensor)
        zf.writestr(f"{b}.npy", buf.getvalue())
        cubes[b] = {
            "shape": tensor.shape,
            "mib": tensor.nbytes / 2**20,
            "z0": z0,
            "dz": dz,
            "cells": int(mask.astype(bool).sum()),
        }

exact = [b for b, c in cubes.items() if c["dz"] == 1.0]
print(f"{len(cubes)} blocks -> {len(cubes)} cubes of {next(iter(cubes.values()))['shape']}")
print(
    f"  {len(exact)} at the requested 1 m bins; "
    f"{len(cubes) - len(exact)} degraded to fit their relief in 128 bins"
)
for b, c in sorted(cubes.items())[:5]:
    print(
        f"  {b}  z = {c['z0']:8.1f} m + bin * {c['dz']:.2f} m   "
        f"{c['cells']:5,} populated cells   {c['mib']:.1f} MiB"
    )

print(f"saved {path} — reload with np.load(path)['<block>']")

Five functions, two libraries, one polygon — coverage, shards, timings,
the paired 3-D view, and tensors on disk.